### tabel1: trans_obl_isik -> transaction + kääne + isik(alati/mitte kunagi)
### tabel2: trans_isik_cnt -> sõna + lemma count + elus count + mitte elus count

In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import powerlaw as pwl
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import os
from common_sql import update_table, create_count_table, create_left_join_table

In [ ]:
# transaktsioonide andmebaas
transaction_db = "../example_data/v33_subset.db"

# loodavad tabelid
vp_data_db = "../example_data/vp_data_actors.db"

# verbimustrite andmebaas
pattern_db = "../example_data/verb_patterns_actors.db"


In [2]:
con = sqlite3.connect(vp_data_db)
cur = con.cursor()
cur.execute(f'ATTACH DATABASE "{transaction_db}" AS trans')
cur.execute(f'ATTACH DATABASE "{pattern_db}" AS isikud')

In [ ]:
# define table names

trans_head = "transaction_head"
transactions = "transaction_v2"

# uus tabel, ainult obl transactions
obl_transactions = "transactions_verbs_wcomps_obl"
# obl transactions mis on alati mustritega
obl_transactions_alati = "transactions_verbs_wcomps_obl_alati"
# obl transactions mis on 'mitte kunagi' mustritega
obl_transactions_mitte_kunagi = "transactions_verbs_wcomps_obl_mitte_kunagi"

# mustrite tabelid
isikud_patterns_alati = "isikud.patterns_actors_len1_alati"
isikud_patterns_mitte_kunagi = "isikud.patterns_actors_len1_mitte_kunagi"

# temp ja uus tabel kus on lisaks veerud w_case ja isik
trans_obl_isik_tmp = "transactions_verbs_wcomps_obl_isik_tmp"
trans_obl_isik = "transactions_verbs_wcomps_obl_isik"

# sõna + root_cnt, elus_cnt ja mitte_elus_cnt abitabelid ja lõplik tabel
trans_obl_isik_counts1 = "transactions_verbs_wcomps_obl_isik_root_eluskoht_counts_base"
trans_obl_isik_counts_elus = "transactions_verbs_wcomps_obl_isik_root_eluskoht_counts_elus"
trans_obl_isik_counts_koht = "transactions_verbs_wcomps_obl_isik_root_eluskoht_counts_koht"
trans_isik_cnt1 = "transactions_verbs_wcomps_obl_isik_root_eluskoht_counts_1"
trans_isik_cnt = "transactions_verbs_wcomps_obl_isik_root_eluskoht_counts"


#### transactions tabelist sõnad, mis on seotud annotatsiooniga 

- alati isikumäärus -> isik
- mitte kunagi isikumäärus -> mitte isik (eeldatavalt koht/sündmus)



In [3]:
%%time

cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=obl_transactions))

cur.execute("""
Create table {new_table} as
SELECT distinct
    tr.head_id as head_id,
    tbl1.verb as verb,
    tbl1.verb_compound as verb_compound,
    tr.id as transaction_id,
    tr.lemma as root_word,
    tr.deprel as word_deprel,
    tr.pos as pos,
    tr.feats as tr_feats,
    tr.koht as koht,
    tr.elus as elus

FROM {trans_head} as tbl1
join {trans} as tr
on tbl1.id = tr.head_id
where tr.deprel = 'obl'
""".format(new_table=obl_transactions, trans_head=trans_head, trans=transactions))


CPU times: user 30.2 s, sys: 8.06 s, total: 38.3 s
Wall time: 45.6 s


### võtta "alati" isikud verbidega seotud sõnad

In [33]:
%%time

cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=obl_transactions_alati))

cur.execute("""
Create table {new_table} as
SELECT distinct
verbs.verb, verbs.verb_compound, verbs.root_word, verbs.koht, verbs.elus,tbl1.w_case, 'alati' as isik
from {obl_trans} as verbs
join {alati_pat} as tbl1
on tbl1.verb_word = verbs.verb
and tbl1.compound_prt1 = verbs.verb_compound
and INSTR(',' || verbs.tr_feats || ',', ',' || tbl1.w_case || ',') > 0
""".format(new_table=obl_transactions_alati, obl_trans=obl_transactions, alati_pat = isikud_patterns_alati))

CPU times: user 11 s, sys: 536 ms, total: 11.6 s
Wall time: 11.7 s


### võtta "mitte kunagi" isikud verbidega seotud sõnad

In [34]:
%%time

cur.execute("""DROP TABLE IF EXISTS {tbl}""".format(tbl=obl_transactions_mitte_kunagi))

cur.execute("""
Create table {new_table} as
SELECT distinct
verbs.verb, verbs.verb_compound, verbs.root_word, verbs.koht, verbs.elus,tbl1.w_case, 'mitte kunagi' as isik
from {obl_trans} as verbs
join {mk_pat} as tbl1
on tbl1.verb_word = verbs.verb
and tbl1.compound_prt1 = verbs.verb_compound
and INSTR(',' || verbs.tr_feats || ',', ',' || tbl1.w_case || ',') > 0
""".format(new_table=obl_transactions_mitte_kunagi, onl_trans=obl_transactions, mk_pat = isikud_patterns_mitte_kunagi))

CPU times: user 46.2 s, sys: 10.7 s, total: 56.9 s
Wall time: 57.4 s


## transactions obl tabel, kus on juures isik = alati/mitte kunagi

In [ ]:
create_left_join_table(con, source_tbl1=obl_transactions, source_tbl2=obl_transactions_alati,result_table=trans_obl_isik_tmp,
                selected_columns=["tbl1.*", "tbl2.w_case", "tbl2.isik"],
                condition="tbl1.verb = tbl2.verb and tbl1.verb_compound = tbl2.verb_compound and tbl1.root_word = tbl2.root_word and INSTR(',' || tbl1.tr_feats || ',', ',' || tbl2.w_case || ',') > 0")

create_left_join_table(con, source_tbl1=trans_obl_isik_tmp, source_tbl2=obl_transactions_mitte_kunagi,result_table=trans_obl_isik,
                selected_columns=["tbl1.*","tbl2.w_case as w_case2", "tbl2.isik as isik2"],
                condition="tbl1.verb = tbl2.verb and tbl1.verb_compound = tbl2.verb_compound and tbl1.root_word = tbl2.root_word and INSTR(',' || tbl1.tr_feats || ',', ',' || tbl2.w_case || ',') > 0")


In [ ]:
cur.execute("""
ALTER TABLE {tbl} ADD isikumaarus text NOT NULL DEFAULt('')""".format(tbl=trans_obl_isik))
con.commit()

update_table(con, trans_obl_isik, "isik", "'mitte kunagi'", "isik2 = 'mitte kunagi'")
update_table(con, trans_obl_isik, "w_case", "w_case2", "w_case2 is not null")

In [50]:
query = """SELECT * from {tbl} limit 10""".format(tbl=trans_obl_isik)

source2 = pd.read_sql_query(query, con)
source2

,head_id,verb,verb_compound,transaction_id,root_word,word_deprel,pos,tr_feats,koht,elus,w_case,isik,w_case2,isik2
0,2,toimuma,,1,lõpp,obl,S,"com,in,sg",UNK,UNK,in,mitte kunagi,in,mitte kunagi
1,2,toimuma,,3,1.,obl,N,"<?>,ord,roman",UNK,UNK,None,None,None,None
2,3,saama,pihta,7,keel,obl,S,"all,com,pl",UNK,UNK,None,None,None,None
3,6,kulmineeruma,,15,purukspeksmine,obl,S,"com,kom,sg",UNK,UNK,None,None,None,None
4,10,tulema,,19,sina,obl,P,"ad,sg",UNK,YES,ad,alati,None,None
5,11,viilima,,22,tund,obl,S,"com,el,pl",UNK,UNK,None,None,None,None
6,11,viilima,,23,juht,obl,S,"ad,com,sg",UNK,YES,None,None,None,None
7,16,tulema,,25,mis,obl,P,"gen,pl",UNK,UNK,None,None,None,None
8,23,alustama,,36,muusika,obl,S,"com,kom,sg",UNK,UNK,None,None,None,None
9,25,muutuma,,40,mis,obl,P,"el,sg",UNK,UNK,None,None,None,None


## Create necessary tables for plotting

In [64]:
# base table with root counts
cur.execute("""drop table if exists {tbl}""".format(tbl=trans_obl_isik_counts1))

cur.execute("""
create table {new_table} as
SELECT root_word, count(root_word) as root_cnt
from {isik}
group by root_word
""".format(new_table=trans_obl_isik_counts1, isik=trans_obl_isik))

# table counts elus 
create_count_table(con, trans_obl_isik, trans_obl_isik_counts_elus,
                  ["root_word"], "root_word", "elus_cnt", "isik = 'alati'", ["root_word"])

# tbl count koht
create_count_table(con, trans_obl_isik, trans_obl_isik_counts_koht,
                  ["root_word"], "root_word", "mitte_elus_cnt", "isik = 'mitte kunagi'", ["root_word"])


In [70]:
# join everything into 1 table

create_left_join_table(con, source_tbl1=trans_obl_isik_counts1, source_tbl2=trans_obl_isik_counts_elus, 
                result_table=trans_isik_cnt1,
                selected_columns=["tbl1.root_word", "tbl1.root_cnt", "elus_cnt"],
                condition="tbl1.root_word=tbl2.root_word ")

create_left_join_table(con, source_tbl1=trans_isik_cnt1, source_tbl2=trans_obl_isik_counts_koht, 
                result_table=trans_isik_cnt,
                selected_columns=["tbl1.root_word", "tbl1.root_cnt", "tbl1.elus_cnt", "mitte_elus_cnt"],
                condition="tbl1.root_word=tbl2.root_word ")

In [71]:
# update table null -> 0
update_table(con, trans_isik_cnt, "elus_cnt", 0, "elus_cnt is null")
update_table(con, trans_isik_cnt, "mitte_elus_cnt", 0, "mitte_elus_cnt is null")

In [74]:
query = """
SELECT * from  {tbl} where elus_cnt!=mitte_elus_cnt limit 20 """.format(tbl=trans_isik_cnt)

source2 = pd.read_sql_query(query, con)
source2

,root_word,root_cnt,elus_cnt,mitte_elus_cnt
0,$,321,0,1
1,$1,2,0,1
2,%,32,0,4
3,%-ilis,1,0,1
4,%-põhimõte,1,0,1
5,%-see,3,0,2
6,%line,54,8,34
7,-30s,1,0,1
8,-4%,17,0,2
9,-4.,3,0,1


In [82]:
query = """SELECT * from {tbl}""".format(tbl=trans_isik_cnt)
s = pd.read_sql_query(query, con)
s.to_csv(trans_isik_cnt+".csv", sep=",", index=False, encoding="utf-8")

In [4]:
con.close()